First, we need to import our camera class. The fast camera driver classes are located within the `jisa` package which first needs to be initialised by running `import pyjisa.autoload`, then we can import what we need.

In [7]:
# FastCamera classes are located within the jisa package which first needs initialising
import pyjisa.autoload

# Now we can import our camera class(es)
from jisa.devices.camera import Andor2, Andor3, Lumenera, ThorCam, FakeCamera, Camera
from jisa.devices.camera.feature import *
from jisa.devices.camera.imagemodes import *
from jisa.devices.features import *

import numpy as np

Now, to connect to your camera, instantiate an object of the correct class, giving it whatever parameter that camera needs.

In [ ]:
newton = Andor2(0)           # Andor Newton CCD
zyla   = Andor3(1)           # Andor Zyla sCMOS
thor   = ThorCam.Colour()    # ThorLabs colour camera
lumen  = Lumenera(1)         # Lumenera camera

You can configure your camera using the `set...()` methods it provides.

In [ ]:
camera: Camera = ... # Whichever camera we are using

# Exposure time 10 us
camera.setIntegrationTime(10e-6)

# Use region-of-interest mode, only using 512x128 pixels, auto centred
camera.setImageMode(Camera.ImageMode.ROI)
camera.setImageWidth(512)
camera.setImageHeight(128)
camera.setImageCentredX(True)
camera.setImageCentredY(True)

# Bin pixels into 2x2 bins
camera.setBinningX(2)
camera.setBinningY(2)

# Some cameras are temperature controlled
if isinstance(camera, TemperatureControlled):

    camera.setTemperatureControlTarget(183.0)
    camera.setTemperatureControlEnabled(True)
    camera.waitForTemperatureControlStable()

# Some have a configurable preamp gain
if isinstance(camera, Amplified):
    camera.setAmplifierGain(25.0)

# CMOS cameras have variable readout rates
if isinstance(camera, CMOS):
    rates = camera.getPixelReadoutRates()
    camera.setPixelReadoutRate(np.max(rates)) # Set to max rate

# Some cameras can timestamp their frames using an internal clock
if isinstance(camera, Timestamping):
    camera.setHardwareTimestampingEnabled(True)

You can ask the camera for individual frames, or a series of frames synchronously

In [ ]:
frame  = newton.getFrame()         # Takes one frame then returns it
frames = newton.getFrameSeries(10) # Takes 10 frames then returns in a list